In [1]:
%pip install -q numpy==2.4.4 pandas==3.0.2 scikit-learn==1.8.0 joblib==1.5.3

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 03 - Huấn luyện tuần tự và so sánh model

Notebook nhận đúng split và feature contract từ `02_preprocess.ipynb`. Các model được huấn luyện **từng cái một** trên cùng train/test, cùng `StratifiedKFold(n_splits=5)`, cùng Pipeline tiền xử lý và cùng metric GridSearch là Recall của lớp 1.

## Baseline và bốn model

| Model | Vai trò | Tham số thử |
|---|---|---|
| **Dummy baseline** | Mốc tham chiếu, luôn dự đoán lớp phổ biến nhất; không dùng để triển khai. | `strategy='most_frequent'` |
| **Logistic Regression** | Baseline tuyến tính; dễ giải thích, nhẹ và nhanh. `class_weight='balanced'` giúp giảm ảnh hưởng mất cân bằng. | `C`: 0.01, 0.1, 1, 10 |
| **Gaussian Naive Bayes** | Model xác suất; giả định các feature độc lập có điều kiện theo lớp. | `var_smoothing`: 1e-9, 1e-8, 1e-7 |
| **SVM RBF** | Tìm biên phân tách phi tuyến bằng kernel RBF. | `C`: 1, 10; `gamma`: `scale`, `auto` |
| **Random Forest** | Ensemble nhiều cây, bắt quan hệ phi tuyến nhưng file lớn hơn. | `n_estimators`: 200; `max_depth`: 6, 12, None |

Mỗi model được huấn luyện và đánh giá riêng, sau đó kết quả được gom vào bảng so sánh. Dummy baseline chỉ giúp biết các model học được tốt hơn mốc đơn giản đến mức nào; nó bị loại khỏi bước chọn model production.

In [2]:
%pip install -q pandas numpy scikit-learn joblib

from pathlib import Path
from zipfile import ZipFile
import json
import time
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

SEARCH_ROOTS = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((root for root in SEARCH_ROOTS if (root / 'ai-models').is_dir()), Path.cwd())
WORK_DIR = Path('/content') if Path('/content').exists() else PROJECT_ROOT / '.colab_artifacts'
WORK_DIR.mkdir(parents=True, exist_ok=True)
split_path = WORK_DIR / 'preprocessed_split.joblib'
if split_path.exists():
    split = joblib.load(split_path)
    print(f'Loaded preprocessing artifact: {split_path}')
else:
    print('Khong tim thay preprocessed_split.joblib; tao split tu dataset.zip...')
    DATA_NAME = 'healthcare-dataset-stroke-data.csv'
    DATA_ROOTS = []
    for root in SEARCH_ROOTS:
        DATA_ROOTS.extend([root / 'ai-models' / 'data', root / 'data'])
    zip_candidates = [root / 'dataset.zip' for root in DATA_ROOTS]
    zip_path = next((path for path in zip_candidates if path.exists()), None)
    if zip_path is None:
        try:
            from google.colab import files
        except ImportError as error:
            raise FileNotFoundError('Khong tim thay dataset.zip. Dat file vao ai-models/data/ hoac upload tren Colab.') from error
        uploaded = files.upload()
        uploaded_path = Path('/content') / next(iter(uploaded))
        if uploaded_path.name != 'dataset.zip':
            raise ValueError('Vui long upload dung file dataset.zip.')
        zip_path = uploaded_path
    with ZipFile(zip_path) as archive:
        csv_names = [name for name in archive.namelist() if Path(name).name == DATA_NAME]
        if not csv_names:
            raise FileNotFoundError(f'{DATA_NAME} khong co trong {zip_path}')
        with archive.open(csv_names[0]) as csv_file:
            data = pd.read_csv(csv_file)
    data = data.drop_duplicates().drop(columns=['id'], errors='ignore')
    features = ['age', 'avg_glucose_level', 'bmi', 'hypertension', 'heart_disease', 'gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
    X = data[features]
    y = data['stroke']
    split = {}
    split['X_train'], split['X_test'], split['y_train'], split['y_test'] = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    split['feature_names'] = features
    joblib.dump(split, split_path, compress=3)
    print(f'Created preprocessing artifact: {split_path}')

X_train, X_test = split['X_train'], split['X_test']
y_train, y_test = split['y_train'], split['y_test']
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Positive rate train: {y_train.mean():.2%} | test: {y_test.mean():.2%}')

NUMERIC = ['age', 'avg_glucose_level', 'bmi']
BINARY = ['hypertension', 'heart_disease']
CATEGORICAL = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
FEATURES = NUMERIC + BINARY + CATEGORICAL
if list(X_train.columns) != FEATURES:
    raise ValueError('Feature contract khong khop voi 02_preprocess.ipynb.')

def build_preprocessor():
    return ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), NUMERIC),
        ('bin', Pipeline([('imputer', SimpleImputer(strategy='most_frequent'))]), BINARY),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), CATEGORICAL),
    ])

models = {
    'dummy_baseline': (DummyClassifier(strategy='most_frequent'), {}),
    'logistic_regression': (LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42), {'clf__C': [0.01, 0.1, 1, 10]}),
    'naive_bayes': (GaussianNB(), {'clf__var_smoothing': [1e-9, 1e-8, 1e-7]}),
    'svm_rbf': (SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42), {'clf__C': [1, 10], 'clf__gamma': ['scale', 'auto']}),
    'random_forest': (RandomForestClassifier(class_weight='balanced', random_state=42), {'clf__n_estimators': [200], 'clf__max_depth': [6, 12, None]}),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models_dir = WORK_DIR / 'models' / 'candidates'
models_dir.mkdir(parents=True, exist_ok=True)
print(f'Models run tuần tự: {", ".join(models)}')


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Loaded preprocessing artifact: \content\preprocessed_split.joblib
Train: (4088, 10), Test: (1022, 10)
Positive rate train: 4.99% | test: 4.99%
Models run tuần tự: dummy_baseline, logistic_regression, naive_bayes, svm_rbf, random_forest


In [3]:
results = []

def train_one_model(name):
    estimator, params = models[name]
    print(f'===== Training: {name} =====')
    pipe = Pipeline([('preprocess', build_preprocessor()), ('clf', estimator)])
    train_started = time.perf_counter()
    search = GridSearchCV(pipe, params, scoring='recall', cv=cv, n_jobs=-1, refit=True)
    search.fit(X_train, y_train)
    train_seconds = time.perf_counter() - train_started
    best_pipe = search.best_estimator_

    predict_started = time.perf_counter()
    y_pred = best_pipe.predict(X_test)
    y_proba = best_pipe.predict_proba(X_test)[:, 1]
    inference_ms = (time.perf_counter() - predict_started) / len(X_test) * 1000

    model_path = models_dir / f'{name}.joblib'
    joblib.dump(best_pipe, model_path, compress=3)
    result = {
        'model': name,
        'model_type': 'baseline' if name == 'dummy_baseline' else 'candidate',
        'best_params': search.best_params_,
        'cv_best_recall': float(round(search.best_score_, 4)),
        'cv_recall_std': float(round(search.cv_results_['std_test_score'][search.best_index_], 4)),
        'train_seconds': round(train_seconds, 3),
        'recall_class1': round(recall_score(y_test, y_pred), 4),
        'roc_auc': round(roc_auc_score(y_test, y_proba), 4),
        'precision_class1': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'f1_class1': round(f1_score(y_test, y_pred, zero_division=0), 4),
        'inference_ms_per_sample': round(inference_ms, 3),
        'file_size_kb': round(model_path.stat().st_size / 1024, 1),
        'model_path': str(model_path),
    }
    results.append(result)
    print(f"{name}: Recall={result['recall_class1']} | ROC-AUC={result['roc_auc']} | F1={result['f1_class1']}")
    return result

In [4]:
dummy_result = train_one_model('dummy_baseline')
display(pd.DataFrame([dummy_result]))

===== Training: dummy_baseline =====
dummy_baseline: Recall=0.0 | ROC-AUC=0.5 | F1=0.0


,model,model_type,best_params,cv_best_recall,cv_recall_std,train_seconds,recall_class1,roc_auc,precision_class1,f1_class1,inference_ms_per_sample,file_size_kb,model_path
0,dummy_baseline,baseline,{},0.0,0.0,3.175,0.0,0.5,0.0,0.0,0.017,2.0,\content\models\candidates\dummy_baseline.joblib


In [5]:
logistic_result = train_one_model('logistic_regression')
display(pd.DataFrame([logistic_result]))

===== Training: logistic_regression =====
logistic_regression: Recall=0.8039 | ROC-AUC=0.8819 | F1=0.296


,model,model_type,best_params,cv_best_recall,cv_recall_std,train_seconds,recall_class1,roc_auc,precision_class1,f1_class1,inference_ms_per_sample,file_size_kb,model_path
0,logistic_regression,candidate,{'clf__C': 0.01},0.8482,0.0518,2.073,0.8039,0.8819,0.1814,0.296,0.015,2.3,\content\models\candidates\logistic_regression...


In [6]:
naive_bayes_result = train_one_model('naive_bayes')
display(pd.DataFrame([naive_bayes_result]))

===== Training: naive_bayes =====
naive_bayes: Recall=1.0 | ROC-AUC=0.8832 | F1=0.1096


,model,model_type,best_params,cv_best_recall,cv_recall_std,train_seconds,recall_class1,roc_auc,precision_class1,f1_class1,inference_ms_per_sample,file_size_kb,model_path
0,naive_bayes,candidate,{'clf__var_smoothing': 1e-09},1.0,0.0,0.212,1.0,0.8832,0.058,0.1096,0.017,2.6,\content\models\candidates\naive_bayes.joblib


In [7]:
svm_result = train_one_model('svm_rbf')
display(pd.DataFrame([svm_result]))

===== Training: svm_rbf =====
svm_rbf: Recall=0.7843 | ROC-AUC=0.8757 | F1=0.292


,model,model_type,best_params,cv_best_recall,cv_recall_std,train_seconds,recall_class1,roc_auc,precision_class1,f1_class1,inference_ms_per_sample,file_size_kb,model_path
0,svm_rbf,candidate,"{'clf__C': 1, 'clf__gamma': 'auto'}",0.8434,0.0624,8.227,0.7843,0.8757,0.1794,0.292,0.414,54.7,\content\models\candidates\svm_rbf.joblib


In [8]:
random_forest_result = train_one_model('random_forest')
display(pd.DataFrame([random_forest_result]))

===== Training: random_forest =====
random_forest: Recall=0.6863 | ROC-AUC=0.8821 | F1=0.3535


,model,model_type,best_params,cv_best_recall,cv_recall_std,train_seconds,recall_class1,roc_auc,precision_class1,f1_class1,inference_ms_per_sample,file_size_kb,model_path
0,random_forest,candidate,"{'clf__max_depth': 6, 'clf__n_estimators': 200}",0.7111,0.0669,4.022,0.6863,0.8821,0.2381,0.3535,0.071,542.1,\content\models\candidates\random_forest.joblib


In [9]:
comparison_path = WORK_DIR / 'models' / 'comparison.json'
with open(comparison_path, 'w', encoding='utf-8') as file:
    json.dump(results, file, ensure_ascii=False, indent=2)

comparison_table = pd.DataFrame(results).sort_values(['recall_class1', 'roc_auc'], ascending=False)
display(comparison_table)
print(f'Saved models under {WORK_DIR / "models"}')
print('Đã train riêng từng model; cell này chỉ tổng hợp và so sánh kết quả.')

,model,model_type,best_params,cv_best_recall,cv_recall_std,train_seconds,recall_class1,roc_auc,precision_class1,f1_class1,inference_ms_per_sample,file_size_kb,model_path
2,naive_bayes,candidate,{'clf__var_smoothing': 1e-09},1.0000,0.0000,0.212,1.0000,0.8832,0.0580,0.1096,0.017,2.6,\content\models\candidates\naive_bayes.joblib
1,logistic_regression,candidate,{'clf__C': 0.01},0.8482,0.0518,2.073,0.8039,0.8819,0.1814,0.2960,0.015,2.3,\content\models\candidates\logistic_regression...
3,svm_rbf,candidate,"{'clf__C': 1, 'clf__gamma': 'auto'}",0.8434,0.0624,8.227,0.7843,0.8757,0.1794,0.2920,0.414,54.7,\content\models\candidates\svm_rbf.joblib
4,random_forest,candidate,"{'clf__max_depth': 6, 'clf__n_estimators': 200}",0.7111,0.0669,4.022,0.6863,0.8821,0.2381,0.3535,0.071,542.1,\content\models\candidates\random_forest.joblib
0,dummy_baseline,baseline,{},0.0000,0.0000,3.175,0.0000,0.5000,0.0000,0.0000,0.017,2.0,\content\models\candidates\dummy_baseline.joblib


Saved models under \content\models
Đã train riêng từng model; cell này chỉ tổng hợp và so sánh kết quả.


In [10]:
from datetime import datetime, timezone
from shutil import copyfile
from zipfile import ZipFile
import numpy as np
import sklearn

MIN_PRECISION = 0.15
candidate_results = [result for result in results if result['model_type'] == 'candidate']
eligible = [result for result in candidate_results if result['precision_class1'] >= MIN_PRECISION]
selected_pool = eligible or candidate_results
selected = sorted(selected_pool, key=lambda result: (result['recall_class1'], result['roc_auc']), reverse=True)[0]

canonical_models_dir = Path('/content/models') if Path('/content').exists() else PROJECT_ROOT / 'ai-models' / 'models'
canonical_candidates_dir = canonical_models_dir / 'candidates'
canonical_candidates_dir.mkdir(parents=True, exist_ok=True)

def copy_if_needed(source, destination):
    if Path(source).resolve() != Path(destination).resolve():
        copyfile(source, destination)

for result in results:
    copy_if_needed(models_dir / f"{result['model']}.joblib", canonical_candidates_dir / f"{result['model']}.joblib")
copy_if_needed(models_dir / f"{selected['model']}.joblib", canonical_models_dir / 'model.joblib')
copy_if_needed(comparison_path, canonical_models_dir / 'comparison.json')

training_metadata = {
    'model_name': selected['model'],
    'selected_reason': 'Dummy baseline chỉ để tham chiếu; trong các model candidate, lọc precision_class1 >= 0.15 rồi chọn recall_class1 cao nhất với roc_auc tie-breaker.',
    'selection_rule': {
        'baseline_model': 'dummy_baseline',
        'minimum_precision_class1': MIN_PRECISION,
        'primary_metric': 'recall_class1',
        'tie_breaker': 'roc_auc',
    },
    'features': FEATURES,
    'target': 'stroke',
    'positive_class': 1,
    'library_versions': {
        'scikit_learn': sklearn.__version__,
        'numpy': np.__version__,
        'pandas': pd.__version__,
    },
    'trained_at_utc': datetime.now(timezone.utc).isoformat(),
    'source_artifact': str(WORK_DIR / 'models'),
    'all_models_compared': [result['model'] for result in results],
}
with open(canonical_models_dir / 'training_metadata.json', 'w', encoding='utf-8') as file:
    json.dump(training_metadata, file, ensure_ascii=False, indent=2)

print(f"Selected model: {selected['model']}")
print(f"Saved production pipeline: {canonical_models_dir / 'model.joblib'}")
print(f"Saved candidates: {canonical_candidates_dir}")
print(f"Saved training metadata: {canonical_models_dir / 'training_metadata.json'}")

if Path('/content').exists():
    download_path = Path('/content/stroke_models.zip')
    with ZipFile(download_path, 'w') as archive:
        for artifact in canonical_models_dir.rglob('*'):
            if artifact.is_file():
                archive.write(artifact, artifact.relative_to(canonical_models_dir.parent))
        split_artifact = WORK_DIR / 'preprocessed_split.joblib'
        if split_artifact.exists():
            archive.write(split_artifact, split_artifact.name)
    print(f'Created download archive: {download_path}')
    try:
        from google.colab import files
        files.download(str(download_path))
    except ImportError:
        print('Download stroke_models.zip manually from the Colab file browser.')
else:
    print(f'Local models are already available at: {canonical_models_dir}')

Selected model: logistic_regression
Saved production pipeline: \content\models\model.joblib
Saved candidates: \content\models\candidates
Saved training metadata: \content\models\training_metadata.json
Created download archive: \content\stroke_models.zip
Download stroke_models.zip manually from the Colab file browser.
